# LLM Cluster Refinement (LLMEdgeRefine-style)

Applies an LLM-based post-clustering refinement stage on top of hard clusters from VAE-BM / BERTopic / FASTopic / GloCOM, and reports before/after ACC, NMI, ARI, AMI, Purity, Silhouette, Davies-Bouldin, Calinski-Harabasz.

**Requires a GPU runtime** (Runtime -> Change runtime type -> GPU) for 4-bit Mistral-7B inference. This notebook was written but **not executed** during development - run each cell yourself and validate the output.

## 1. Clone repo

In [ ]:
REPOSITORY_URL = "https://github.com/<owner>/vaebm-baselines-comparision.git"
!git clone {REPOSITORY_URL}
%cd vaebm-baselines-comparision

## 2. Install dependencies (project + LLM extra)

In [ ]:
!pip install -e ".[experiment,llm]" -q

## 3. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - Mistral-7B 4-bit refinement requires a GPU runtime")

## 4. Print GPU memory

In [ ]:
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_bytes / 1e9:.1f} GB free / {total_bytes / 1e9:.1f} GB total")
else:
    print("No CUDA device.")

## 5. Confirm bitsandbytes/transformers versions

In [ ]:
import transformers, bitsandbytes
print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

## 6. Load dataset

Uses this project's own dataset registry (`src/vaebm_benchmark/datasets/simple_registry.py`) - downloads and caches automatically.

In [ ]:
from vaebm_benchmark.datasets.simple_registry import load_dataset

DATASET = "search_snippets"
documents, labels, num_classes = load_dataset(DATASET)
print(f"{DATASET}: {len(documents)} documents, {num_classes} ground-truth classes (used only to size K)")

## 7. Run baseline clustering (no LLM yet)

Sanity-check the baseline's own hard clusters before spending any LLM budget - reuses `--experiment cluster`'s own model registry.

In [ ]:
!python scripts/run_experiment.py --experiment cluster --models vaebm bertopic fastopic glocom --datasets search_snippets

## 8. Load Mistral-7B (4-bit)

`llm/client.py`'s `LLMClient` is LAZY - it only loads the model on the first `.generate()` call. This cell forces an early load just to confirm it works before the full refinement run.

In [ ]:
from vaebm_benchmark.llm.client import LLMClient

llm_client = LLMClient(model_name="mistralai/Mistral-7B-Instruct-v0.3", quantization="4bit")
test_result = llm_client.generate('Return JSON only: {"cluster_id": 0, "confidence": 0.9}')
print(test_result)

## 9. Run LLM refinement

In [ ]:
!python scripts/run_experiment.py \
  --experiment llm_cluster_refinement \
  --models vaebm bertopic fastopic glocom \
  --datasets search_snippets \
  --llm-model mistralai/Mistral-7B-Instruct-v0.3 \
  --edge-fraction 0.10 \
  --candidate-clusters 3 \
  --min-confidence 0.70 \
  --refinement-iterations 1 \
  --quantization 4bit

## 10. Print before/after table

In [ ]:
import pandas as pd

results_df = pd.read_csv("results/llm_refinement/llm_refinement_results.csv")
display(results_df[["model", "dataset", "acc_before", "acc_after", "delta_acc",
                     "nmi_before", "nmi_after", "delta_nmi",
                     "silhouette_before", "silhouette_after", "delta_silhouette",
                     "davies_bouldin_before", "davies_bouldin_after", "delta_davies_bouldin",
                     "number_of_edge_points", "percentage_documents_sent_to_llm",
                     "number_of_reassignments", "total_tokens"]])

## 11. Inspect reassigned examples

In [ ]:
import json

with open("results/llm_refinement/llm_refinement_results.json") as handle:
    full_results = json.load(handle)

# Per-document decisions are cached, not embedded in the summary JSON above -
# inspect the LLM decision cache directly for reassigned examples.
with open("results/llm_cache/decisions.json") as handle:
    decisions = json.load(handle)

reassigned = [d for d in decisions.values() if d.get("valid") and d.get("cluster_id") is not None]
print(f"{len(decisions)} cached LLM decisions total")
for decision in reassigned[:10]:
    print(decision)

## 12. Save / download results

In [ ]:
!zip -r llm_refinement_results.zip results/llm_refinement results/llm_cache

from google.colab import files
files.download("llm_refinement_results.zip")